# p53 Rescue Mutation Discovery — Full Simulation

This notebook runs the **p53-proteoMgCAD** computational protein design pipeline to discover
second-site rescue mutations that restore tumor suppressor function in mutant p53.

**Pipeline overview:**
1. Environment setup & dependency check
2. Load models (ESM-2 protein language model + functional oracle)
3. Build scenario matrix (8 cancer hotspots x 3 delivery methods)
4. Run campaign (Pass A screening → Pass B deep refinement)
5. Analyze & visualize results (Top-30 shortlist, clinical impact, heatmaps)

## 1. Environment Setup

In [ ]:
import os, sys

# Runtime guards
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Ensure project root is on path
PROJECT_ROOT = os.path.dirname(os.path.abspath("__file__"))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Install package if needed
try:
    import p53cad
    print("p53cad already installed")
except ImportError:
    print("Installing p53cad...")
    !pip install -e . -q
    import p53cad
    print("p53cad installed successfully")

In [ ]:
# Check runtime capabilities
from p53cad.core.runtime import bootstrap_runtime, get_runtime_capabilities

bootstrap_runtime(seed=42)
caps = get_runtime_capabilities()

print("Runtime Capabilities")
print("=" * 50)
for key, val in caps.items():
    print(f"  {key:<35} {val}")

## 2. Inspect Data: Wild-Type p53 & DMS Dataset

In [ ]:
from p53cad.data.dms import P53_WT, get_dms_data

print(f"p53 Wild-Type Sequence ({len(P53_WT)} amino acids):")
# Print in blocks of 60 with position markers
for i in range(0, len(P53_WT), 60):
    chunk = P53_WT[i:i+60]
    print(f"  {i+1:>4}  {chunk}")

print(f"\nLoading DMS data (Giacomelli 2018)...")
dms_df = get_dms_data()
print(f"  Variants: {len(dms_df):,}")
print(f"  Columns: {list(dms_df.columns)}")
dms_df.head()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Visualize DMS score distribution
score_col = [c for c in dms_df.columns if "Nutlin" in c and "Z" in c]
if score_col:
    scores = dms_df[score_col[0]].dropna()
    fig, ax = plt.subplots(1, 1, figsize=(10, 4))
    ax.hist(scores, bins=80, color="#2563EB", alpha=0.7, edgecolor="white")
    ax.axvline(0, color="red", linestyle="--", label="Neutral (Z=0)")
    ax.axvline(-0.5, color="green", linestyle="--", alpha=0.7, label="Functional threshold (Z=-0.5)")
    ax.set_xlabel("Nutlin-3 Z-score")
    ax.set_ylabel("Count")
    ax.set_title("DMS Functional Score Distribution (Giacomelli 2018)")
    ax.legend()
    plt.tight_layout()
    plt.show()
    print(f"  Mean Z-score: {scores.mean():.3f}")
    print(f"  Functional (Z<0): {(scores < 0).sum():,} / {len(scores):,} ({100*(scores < 0).mean():.1f}%)")
else:
    print("Score column not found — skipping visualization")

## 3. Scenario Matrix

The campaign explores rescue mutations for 8 major p53 cancer hotspots (the "BIG8"),
plus pairwise combinations, across 3 delivery methods.

In [ ]:
from p53cad.results.schema import BIG8_HOTSPOTS, DEFAULT_DELIVERY_METHODS, build_scenario_matrix

print("BIG8 Cancer Hotspots:")
for i, h in enumerate(BIG8_HOTSPOTS, 1):
    print(f"  {i}. {h}")

print(f"\nDelivery Methods: {DEFAULT_DELIVERY_METHODS}")

# Build the full scenario matrix
scenarios = build_scenario_matrix(
    hotspots=BIG8_HOTSPOTS,
    delivery_methods=DEFAULT_DELIVERY_METHODS,
    include_pairs=True,
)
print(f"\nTotal scenarios: {len(scenarios)}")
print(f"  Single-hotspot: {sum(1 for s in scenarios if '+' not in s.target_label)}")
print(f"  Pair combos:    {sum(1 for s in scenarios if '+' in s.target_label)}")

# Show first few
print(f"\nFirst 10 scenarios:")
for s in scenarios[:10]:
    print(f"  {s.scenario_id:<40} targets={s.targets}  delivery={s.delivery_method}")

## 4. Run the Campaign

**Budget options:**
| Budget | Pass A Steps | Pass B Steps | Approx. Time |
|--------|-------------|-------------|---------------|
| `fast` | 50 steps x 1 restart | 100 steps x 2 restarts | ~5 min |
| `medium` | 200 steps x 3 restarts | 500 steps x 5 restarts | ~80 min |
| `high` | 500 steps x 8 restarts | 2000 steps x 20 restarts | ~4 hrs |

Change `BUDGET` below to control run time.

In [ ]:
# ===== CONFIGURATION =====
BUDGET = "fast"          # "fast", "medium", or "high"
SEED = 42
INCLUDE_PAIRS = True     # Include pairwise hotspot combos
SHORTLIST_N = 30         # Number of top candidates to shortlist
# =========================

In [ ]:
import time
from p53cad.engine.campaign import CampaignRunner

print(f"Initializing CampaignRunner...")
runner = CampaignRunner()

# Report loaded model info
if runner.embedder is not None:
    print(f"  ESM-2 model: {runner.embedder.model_name}")
    print(f"  Hidden dim:  {runner.embedder.hidden_size}")
if runner.oracle is not None:
    print(f"  Oracle input_dim: {runner.oracle.input_dim}")
    arch = "attention_pooling" if hasattr(runner.oracle.model, 'attn') else "legacy_mlp"
    print(f"  Oracle architecture: {arch}")
print(f"  Pairwise DMS entries: {len(runner._pairwise_dms)}")
print(f"  Contact map entries:  {len(runner._wt_contacts)}")
print(f"\nReady to run.")

In [ ]:
print(f"Starting campaign (budget={BUDGET}, seed={SEED})...")
print(f"This may take a while depending on budget.\n")

t0 = time.time()

result = runner.run(
    budget=BUDGET,
    seed=SEED,
    include_pairs=INCLUDE_PAIRS,
    shortlist_n=SHORTLIST_N,
    with_clinical=True,
)

elapsed = time.time() - t0

print(f"\n{'=' * 60}")
print(f"  CAMPAIGN COMPLETE — {elapsed/60:.1f} min")
print(f"{'=' * 60}")
print(f"  Run ID:      {result['run_id']}")
print(f"  Scenarios:   {result['n_scenarios']}")
print(f"  Candidates:  {result['n_candidates']}")
print(f"  Shortlist:   {result['n_shortlist']}")
print(f"  Run dir:     {result['run_dir']}")

## 5. Results Analysis

In [ ]:
import json
import pandas as pd
import numpy as np

run_dir = result["run_dir"]

# Load all candidates
cand = pd.read_parquet(os.path.join(run_dir, "candidates.parquet"))
print(f"Total candidates: {len(cand)}")
print(f"Columns: {list(cand.columns)}")

# Separate deep-refined candidates
if "pass_name" in cand.columns:
    deep = cand[cand["pass_name"] == "deep"].copy()
    print(f"Deep-refined candidates: {len(deep)}")
else:
    deep = cand.copy()
    print(f"All candidates (no pass separation): {len(deep)}")

cand.describe()

In [ ]:
# Target retention analysis
retains = 0
for _, row in deep.iterrows():
    targets = json.loads(str(row.get("targets_json", "[]")))
    muts = json.loads(str(row.get("mutations_json", "[]")))
    if set(targets).issubset(set(muts)):
        retains += 1

print(f"Target Retention: {retains}/{len(deep)} ({100*retains/max(len(deep),1):.1f}%)")

# Pareto ranking
if "pareto_rank" in deep.columns:
    rank1 = (deep["pareto_rank"] == 1).sum()
    max_rank = int(deep["pareto_rank"].replace([np.inf], np.nan).dropna().max())
    print(f"Pareto rank-1 (non-dominated): {rank1}")
    print(f"Total Pareto fronts: {max_rank}")

# DMS quality
if "rescue_dms_mean" in deep.columns:
    valid_dms = deep[deep["rescue_dms_mean"].notna() & (deep["rescue_dms_mean"] != 0)]
    print(f"\nCandidates with DMS data: {len(valid_dms)}")
    if len(valid_dms):
        print(f"  Mean rescue DMS Z: {valid_dms['rescue_dms_mean'].mean():.3f}")
        func = (valid_dms["rescue_dms_mean"] < 0).sum()
        print(f"  Functional rescues (Z<0): {func}/{len(valid_dms)} ({100*func/len(valid_dms):.0f}%)")

# Score stats
if "score" in deep.columns:
    print(f"\nOracle Score Stats:")
    print(f"  Mean: {deep['score'].mean():.4f}")
    print(f"  Max:  {deep['score'].max():.4f}")
    print(f"  Min:  {deep['score'].min():.4f}")

### 5a. Top-30 Shortlist

In [ ]:
from p53cad.results.schema import select_presentation_shortlist

top = select_presentation_shortlist(deep, top_n=SHORTLIST_N)
print(f"Shortlist: {len(top)} candidates\n")

# Build display table
display_rows = []
for _, row in top.iterrows():
    targets = json.loads(str(row.get("targets_json", "[]")))
    muts = json.loads(str(row.get("mutations_json", "[]")))
    rescue = [m for m in muts if m not in targets]
    dms_z = row.get("rescue_dms_mean", None)
    dms_str = f"{dms_z:+.2f}" if dms_z and dms_z != 0 else "N/A"
    pr = row.get("pareto_rank", None)
    pr_str = f"{int(pr)}" if pr and pr != np.inf else "?"
    display_rows.append({
        "Rank": int(row.get("presentation_rank", 0)),
        "Target": row["target_label"],
        "Rescue Mutations": "+".join(rescue),
        "Oracle Score": f"{float(row['score']):.3f}",
        "DMS Z-score": dms_str,
        "Pareto Rank": pr_str,
        "Delivery": row["delivery_method"],
    })

shortlist_df = pd.DataFrame(display_rows)
shortlist_df

In [ ]:
# Delivery method distribution
if len(top):
    delivery_counts = top["delivery_method"].value_counts()
    print("Delivery Distribution:")
    for d, c in delivery_counts.items():
        print(f"  {d}: {c}")

    unique_targets = top["target_label"].nunique()
    print(f"\nUnique target combos: {unique_targets}")

### 5b. Score Distribution Plots

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1. Oracle score distribution
if "score" in deep.columns:
    axes[0].hist(deep["score"].dropna(), bins=40, color="#2563EB", alpha=0.7, edgecolor="white")
    axes[0].set_title("Oracle Score Distribution")
    axes[0].set_xlabel("Oracle Score")
    axes[0].set_ylabel("Count")

# 2. DMS rescue quality
if "rescue_dms_mean" in deep.columns:
    valid = deep["rescue_dms_mean"].dropna()
    valid = valid[valid != 0]
    if len(valid):
        axes[1].hist(valid, bins=40, color="#10B981", alpha=0.7, edgecolor="white")
        axes[1].axvline(0, color="red", linestyle="--", alpha=0.7)
        axes[1].set_title("DMS Rescue Z-score")
        axes[1].set_xlabel("Z-score")
        axes[1].set_ylabel("Count")

# 3. Scores by target
if "target_label" in deep.columns and "score" in deep.columns:
    # Get single-hotspot targets only for readability
    singles = deep[~deep["target_label"].str.contains(r"\+", na=False)].copy()
    if len(singles):
        target_order = singles.groupby("target_label")["score"].median().sort_values(ascending=False).index
        sns.boxplot(data=singles, x="target_label", y="score", order=target_order,
                    ax=axes[2], palette="Blues_d")
        axes[2].set_title("Score by Hotspot")
        axes[2].set_xlabel("")
        axes[2].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

### 5c. Mutation Position Heatmap

In [ ]:
from p53cad.results.visualization import build_candidate_position_heatmap

# Prepare candidate dicts for heatmap
cand_dicts = []
for _, row in top.iterrows():
    muts = json.loads(str(row.get("mutations_json", "[]")))
    cand_dicts.append({
        "candidate_id": row.get("presentation_rank", 0),
        "profile": row.get("profile", "Unknown"),
        "score": float(row.get("score", 0)),
        "mutations": muts,
    })

matrix_df, freq_df = build_candidate_position_heatmap(cand_dicts)

if not matrix_df.empty:
    fig, axes = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={"height_ratios": [3, 1]})

    # Heatmap
    pos_cols = [c for c in matrix_df.columns if c.startswith("pos_")]
    heatmap_data = matrix_df[pos_cols].values
    pos_labels = [c.replace("pos_", "") for c in pos_cols]

    axes[0].imshow(heatmap_data, aspect="auto", cmap="YlOrRd", interpolation="nearest")
    axes[0].set_xticks(range(len(pos_labels)))
    axes[0].set_xticklabels(pos_labels, rotation=90, fontsize=8)
    axes[0].set_ylabel("Candidate")
    axes[0].set_title("Mutation Position Heatmap (Top-30 Shortlist)")

    # Frequency bar chart
    if not freq_df.empty:
        axes[1].bar(range(len(freq_df)), freq_df["frequency"], color="#2563EB", alpha=0.7)
        axes[1].set_xticks(range(len(freq_df)))
        axes[1].set_xticklabels(freq_df["position"].astype(str), rotation=90, fontsize=8)
        axes[1].set_ylabel("Frequency")
        axes[1].set_xlabel("Position")
        axes[1].set_title("Mutation Frequency by Position")

    plt.tight_layout()
    plt.show()
else:
    print("No mutation position data available for heatmap")

### 5d. Clinical Impact

In [ ]:
from p53cad.analysis.clinical_impact import TCGA_P53_MUTATIONS, CANCER_INCIDENCE, P53_MUTATION_RATE

# Show TCGA hotspot frequencies
print("TCGA p53 Hotspot Mutation Frequencies")
print("=" * 55)
for mut in BIG8_HOTSPOTS:
    info = TCGA_P53_MUTATIONS.get(mut, {})
    freq = info.get("frequency", 0)
    cancers = info.get("cancer_types", [])
    print(f"  {mut:<8}  {freq:>4.1f}%  →  {', '.join(cancers)}")

# Estimate affected patients
print(f"\nEstimated Annual Patient Impact (US):")
print(f"  {'Cancer':<15} {'Incidence':>10} {'p53 mut rate':>12} {'p53 mutant':>12}")
print(f"  {'-'*15} {'-'*10} {'-'*12} {'-'*12}")
total_p53 = 0
for cancer, incidence in sorted(CANCER_INCIDENCE.items(), key=lambda x: -x[1]):
    rate = P53_MUTATION_RATE.get(cancer, 0)
    p53_cases = int(incidence * rate / 100)
    total_p53 += p53_cases
    print(f"  {cancer:<15} {incidence:>10,} {rate:>11}% {p53_cases:>12,}")
print(f"  {'TOTAL':<15} {'':<10} {'':<12} {total_p53:>12,}")

### 5e. Clinical Impact per Shortlisted Candidate

In [ ]:
# Load clinical impact data if available
clinical_path = os.path.join(run_dir, "clinical.parquet")
if os.path.exists(clinical_path):
    clinical_df = pd.read_parquet(clinical_path)
    print(f"Clinical impact records: {len(clinical_df)}")
    clinical_df.head(10)
else:
    print("Clinical impact data not generated in this run.")
    print("(Set with_clinical=True and budget >= 'medium' for clinical analysis)")

## 6. Export Results

In [ ]:
# Export shortlist to CSV
csv_path = os.path.join(run_dir, "top30.csv")
if os.path.exists(csv_path):
    print(f"Shortlist CSV already saved: {csv_path}")
else:
    shortlist_df.to_csv(csv_path, index=False)
    print(f"Shortlist saved to: {csv_path}")

# Summary
summary_path = os.path.join(run_dir, "summary.md")
if os.path.exists(summary_path):
    with open(summary_path) as f:
        print(f.read())
else:
    print("\nNo summary.md generated. Key results:")
    print(f"  Run dir: {run_dir}")
    print(f"  Candidates: {result['n_candidates']}")
    print(f"  Shortlist: {result['n_shortlist']}")

In [ ]:
# List all output artifacts
print(f"\nAll artifacts in {run_dir}:")
for f in sorted(os.listdir(run_dir)):
    fpath = os.path.join(run_dir, f)
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath)
        unit = "KB" if size > 1024 else "B"
        size_str = f"{size/1024:.1f} KB" if size > 1024 else f"{size} B"
        print(f"  {f:<35} {size_str}")
    elif os.path.isdir(fpath):
        n_files = len(os.listdir(fpath))
        print(f"  {f + '/':<35} ({n_files} files)")

---

**Done!** The campaign has completed. Key outputs:
- `candidates.parquet` — All evaluated candidates
- `top30.parquet` / `top30.csv` — Shortlisted rescue mutations
- `clinical.parquet` — Patient impact estimates
- `trajectories.parquet` — Optimization trajectories
- `summary.md` — Human-readable report